# 03 — Routage Intelligent des Tickets (Assignation Agent)
## Helpdesk IA — GEISER | PFE 2026

### Objectif
Prédire automatiquement **quel agent assigner** à un ticket
basé sur l'historique des assignations passées.

### Deux approches comparées
| Approche | Description |
|---|---|
| **ML + Règles hybride** | LinearSVC sur historique + règles catégorie + round-robin |
| **Scoring multi-critères** | Score = (expertise × 2) - charge actuelle |

### Dataset
- **Source** : `dataset_labeled.csv` (produit par notebook 01)
- **Cible** : colonne `assignee` (agent assigné au ticket)

---
## 1. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

print("✅ Imports OK")

---
## 2. Chargement des Données

In [ ]:
# ============================================================
# CHARGEMENT DONNÉES
# ============================================================
df = pd.read_csv('../../data/dataset_labeled.csv', encoding='utf-8')

print(f"✅ Dataset chargé : {df.shape[0]} tickets, {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")

print(f"\n=== AGENTS DISPONIBLES ===")
print(df['assignee'].value_counts().head(20))
print(f"\nNb agents uniques    : {df['assignee'].nunique()}")
print(f"Tickets sans agent   : {df['assignee'].isnull().sum()}")

---
## 3. Analyse des Assignations

Avant d'entraîner le modèle, on analyse la distribution
des tickets par agent pour identifier les agents actifs.

In [ ]:
# ============================================================
# ANALYSE ASSIGNATIONS
# ============================================================
df_assigned = df[df['assignee'].notna()].copy()

print(f"Tickets assignés     : {len(df_assigned)}")
print(f"Tickets non assignés : {df['assignee'].isnull().sum()}")
print(f"Taux assignation     : {len(df_assigned)/len(df)*100:.1f}%")

agent_counts = df_assigned['assignee'].value_counts()

plt.figure(figsize=(14, 6))
agent_counts.head(15).plot(kind='barh', color='#534AB7')
plt.title('Top 15 agents par nombre de tickets traités')
plt.xlabel('Nombre de tickets')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n=== TOP 10 AGENTS ===")
print(agent_counts.head(10))

---
## 4. Préparation du Dataset de Routage

### Stratégie
- On garde uniquement les agents ayant traité **minimum 10 tickets**
- Les agents avec peu de tickets → modèle peu fiable
- On combine : texte + catégorie + priorité comme features

In [ ]:
# ============================================================
# PRÉPARATION DATASET ROUTAGE
# ============================================================

# Garder uniquement les tickets assignés
df_routing = df[df['assignee'].notna()].copy()
df_routing = df_routing.rename(columns={'assignee': 'agent'})

print(f"Tickets assignés : {len(df_routing)}")

# Filtrer agents avec minimum 10 tickets
agent_counts = df_routing['agent'].value_counts()
agents_valides = agent_counts[agent_counts >= 10].index
df_routing = df_routing[df_routing['agent'].isin(agents_valides)].reset_index(drop=True)

print(f"Après filtre (≥10 tickets) : {len(df_routing)} tickets")
print(f"Nb agents valides          : {len(agents_valides)}")
print(f"\nAgents retenus :")
print(agent_counts[agent_counts >= 10])

---
## 5. Feature Engineering

On crée une feature combinée :
**texte + catégorie + priorité** → meilleure représentation

In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================
def clean_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&\w+;', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'[^a-zà-ÿ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Feature combinée : texte + catégorie + priorité
df_routing['feature'] = (
    df_routing['text'].apply(clean_text) + ' ' +
    df_routing['categorie'].fillna('') + ' ' +
    df_routing['priorite'].fillna('')
)

df_routing = df_routing[df_routing['feature'].str.len() > 5].reset_index(drop=True)

print(f"✅ Features créées : {df_routing.shape[0]} tickets")
print(f"\nExemple feature :")
print(df_routing['feature'].iloc[0][:150])
print(f"\nAgent correspondant : {df_routing['agent'].iloc[0]}")

---
## 6. Entraînement du Modèle de Routage

In [ ]:
# ============================================================
# ENTRAÎNEMENT MODÈLE ROUTAGE
# ============================================================

# Features et cible
X = df_routing['feature']
le_agent = LabelEncoder()
y = le_agent.fit_transform(df_routing['agent'])

print(f"Nb classes (agents) : {len(le_agent.classes_)}")
print(f"Agents : {le_agent.classes_.tolist()}")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\n✅ Split :")
print(f"   Train : {len(X_train)} tickets")
print(f"   Test  : {len(X_test)} tickets")

# Modèles à comparer
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, C=1.0
    ),
    'LinearSVC': LinearSVC(
        random_state=42, max_iter=2000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
}

results = {}

print(f"\n=== ENTRAÎNEMENT — ROUTAGE AGENT ===\n")

for name, model in models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )),
        ('clf', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        'pipeline': pipeline,
        'accuracy': acc,
        'f1_score': f1,
        'y_pred': y_pred
    }

    print(f"✅ {name:<25} Accuracy: {acc:.4f}  F1: {f1:.4f}")

In [ ]:
# ============================================================
# MEILLEUR MODÈLE + RAPPORT
# ============================================================
best_name = max(results, key=lambda x: results[x]['f1_score'])
best_model = results[best_name]['pipeline']
y_pred_best = results[best_name]['y_pred']

print(f"🏆 Meilleur modèle : {best_name}")
print(f"   Accuracy        : {results[best_name]['accuracy']:.4f}")
print(f"   F1-Score        : {results[best_name]['f1_score']:.4f}")

print(f"\n=== RAPPORT DÉTAILLÉ ===")
print(classification_report(
    y_test, y_pred_best,
    target_names=le_agent.classes_
))

In [ ]:
# ============================================================
# COMPARAISON VISUELLE
# ============================================================
names = list(results.keys())
accuracies = [results[n]['accuracy'] for n in names]
f1_scores  = [results[n]['f1_score'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, accuracies, width,
               label='Accuracy', color='#534AB7', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_scores, width,
               label='F1-Score', color='#1D9E75', alpha=0.85)

ax.set_title('Comparaison modèles — Routage Agent')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2%}',
            ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2%}',
            ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Règles de Routage par Catégorie

En complément du ML, on définit des règles métier
qui mappent chaque catégorie vers un agent/équipe par défaut.
Utilisé quand la confiance ML est faible.

In [ ]:
# ============================================================
# RÈGLES ROUTAGE PAR CATÉGORIE
# ============================================================

# Analyse : quelle catégorie → quel agent majoritairement
print("=== MAPPING CATÉGORIE → AGENT ===\n")

routing_rules = {}

for cat in df_routing['categorie'].unique():
    df_cat = df_routing[df_routing['categorie'] == cat]
    if len(df_cat) > 0:
        top_agent = df_cat['agent'].value_counts().index[0]
        count = df_cat['agent'].value_counts().iloc[0]
        total = len(df_cat)
        routing_rules[cat] = top_agent
        print(f"  {cat:<20} → {top_agent:<25} "
              f"({count}/{total} = {count/total*100:.0f}%)")

print(f"\n✅ Règles de routage générées pour "
      f"{len(routing_rules)} catégories")

---
## 8. Fonction de Routage Complète

Combinaison des 3 stratégies :
1. ML si confiance ≥ seuil
2. Règles catégorie si ML peu confiant
3. Agent avec moins de tickets (round-robin) en fallback

In [ ]:
# ============================================================
# FONCTION ROUTAGE COMPLÈTE
# ============================================================

def route_ticket(texte, categorie, priorite,
                 agents_charge=None, seuil_confiance=0.6):
    """
    Recommander un agent pour un ticket

    Args:
        texte           : texte du ticket (objet + description)
        categorie       : catégorie prédite
        priorite        : priorité prédite
        agents_charge   : dict {agent: nb_tickets_ouverts}
        seuil_confiance : seuil minimum pour utiliser ML (0.6 = 60%)

    Returns:
        dict : agent recommandé, stratégie utilisée, confiance
    """
    feature = clean_text(texte) + ' ' + categorie + ' ' + priorite

    # === STRATÉGIE 1 : ML si confiance ≥ 60% ===
    agent_enc = best_model.predict([feature])[0]
    agent_ml  = le_agent.inverse_transform([agent_enc])[0]

    try:
        scores     = best_model.decision_function([feature])[0]
        exp_scores = np.exp(scores - scores.max())
        confidence = float(exp_scores.max() / exp_scores.sum())
    except:
        confidence = 0.0

    if confidence >= seuil_confiance:
        return {'agent': agent_ml, 'strategie': 'ML',
                'confidence': f"{confidence:.2%}"}

    # === STRATÉGIE 2 : RÈGLES CATÉGORIE ===
    if categorie in routing_rules:
        return {'agent': routing_rules[categorie],
                'strategie': 'Règle catégorie', 'confidence': 'N/A'}

    # === STRATÉGIE 3 : ROUND-ROBIN ===
    if agents_charge:
        agent_libre = min(agents_charge, key=agents_charge.get)
        return {'agent': agent_libre,
                'strategie': 'Round-robin', 'confidence': 'N/A'}

    return {'agent': le_agent.classes_[0],
            'strategie': 'Fallback', 'confidence': 'N/A'}

# ============================================================
# TESTS
# ============================================================
agents_charge_exemple = {
    agent: np.random.randint(0, 15)
    for agent in le_agent.classes_
}

test_tickets = [
    ("Problème réseau wifi bureau",   "Réseau",             "P2_HIGH"),
    ("Mot de passe outlook oublié",   "Accès & Compte",     "P3_NORMAL"),
    ("Panne ordinateur urgent",       "Matériel",           "P1_URGENT"),
    ("Installation sage paie",        "Logiciel",           "P4_LOW"),
    ("Problème imprimante",           "Matériel",           "P2_HIGH"),
    ("Création compte messagerie",    "Email & Messagerie", "P4_LOW"),
    ("Réclamation client mall sfax",  "Réclamation",        "P3_NORMAL"),
]

print(f"Seuil confiance ML : 60%")
print(f"{'TICKET':<35} {'AGENT':<25} {'STRATÉGIE':<20} {'CONFIANCE'}")
print("-" * 90)

for texte, cat, prio in test_tickets:
    result = route_ticket(texte, cat, prio, agents_charge=agents_charge_exemple)
    print(f"{texte:<35} {result['agent']:<25} "
          f"{result['strategie']:<20} {result['confidence']}")

---
## 9. Approche Alternative — Scoring Multi-Critères

### Principe
Au lieu de prédire directement l'agent avec le ML,
on calcule un **score** pour chaque agent basé sur 2 critères :

| Critère | Poids | Signification |
|---|---|---|
| **Expertise** | × 2 | Nb de tickets de cette catégorie traités historiquement |
| **Charge actuelle** | - 1 | Nb de tickets ouverts actuellement (moins = mieux) |

```
Score(agent) = (expertise × 2) - charge_actuelle
→ L'agent avec le score le plus élevé est sélectionné
```

### Avantages vs ML
- ✅ Transparent : on comprend exactement pourquoi un agent est choisi
- ✅ Prend en compte la charge de travail en temps réel
- ✅ Pas de phase d'entraînement nécessaire
- ✅ S'adapte automatiquement quand les données changent

In [ ]:
# ============================================================
# SCORING MULTI-CRITÈRES
# ============================================================

# Étape 1 : Calculer l'expertise de chaque agent par catégorie
# (nombre de tickets traités historiquement par catégorie)
expertise_matrix = df_routing.groupby(['agent', 'categorie']).size().unstack(fill_value=0)

print("=== MATRICE EXPERTISE (agent × catégorie) ===")
print(expertise_matrix)

# Étape 2 : Simuler la charge actuelle (tickets ouverts)
# En production, ce serait lu depuis la base de données
np.random.seed(42)
charge_actuelle = {agent: np.random.randint(0, 20) for agent in le_agent.classes_}
print(f"\n=== CHARGE ACTUELLE SIMULÉE ===")
for agent, charge in sorted(charge_actuelle.items(), key=lambda x: x[1]):
    print(f"  {agent:<25} : {charge} tickets ouverts")


# Étape 3 : Fonction de scoring
def score_agent(agent, categorie, charge_actuelle):
    """
    Score = (expertise × 2) - charge_actuelle
    Plus le score est élevé, plus l'agent est approprié.
    """
    expertise = 0
    if agent in expertise_matrix.index and categorie in expertise_matrix.columns:
        expertise = expertise_matrix.loc[agent, categorie]
    charge = charge_actuelle.get(agent, 0)
    return (expertise * 2) - charge


def route_by_scoring(categorie, charge_actuelle):
    """Sélectionne l'agent avec le score le plus élevé."""
    scores = {}
    for agent in le_agent.classes_:
        scores[agent] = score_agent(agent, categorie, charge_actuelle)

    best_agent = max(scores, key=scores.get)
    best_score = scores[best_agent]

    return {
        'agent':     best_agent,
        'score':     best_score,
        'strategie': 'Scoring multi-critères',
        'detail':    {a: s for a, s in sorted(scores.items(), key=lambda x: -x[1])[:3]}
    }


# Étape 4 : Tests comparatifs
print(f"\n{'CATÉGORIE':<22} {'AGENT (Scoring)':<25} {'SCORE':<8} {'AGENT (ML+Règles)'}")
print("-" * 85)

categories_test = [
    ("Réseau",             "P2_HIGH",   "Problème réseau wifi bureau"),
    ("Matériel",           "P1_URGENT", "Panne ordinateur urgent"),
    ("Logiciel",           "P4_LOW",    "Installation sage paie"),
    ("Email & Messagerie", "P4_LOW",    "Création compte messagerie"),
    ("Accès & Compte",     "P3_NORMAL", "Mot de passe oublié"),
    ("Réclamation",        "P3_NORMAL", "Réclamation client"),
    ("RH_Finance",         "P4_LOW",    "Nouvel employé insertion"),
]

for cat, prio, texte in categories_test:
    r_score = route_by_scoring(cat, charge_actuelle)
    r_ml    = route_ticket(texte, cat, prio, agents_charge=charge_actuelle)
    print(f"{cat:<22} {r_score['agent']:<25} {r_score['score']:<8} {r_ml['agent']}")

In [ ]:
# ============================================================
# SAUVEGARDE MODÈLE ROUTAGE
# ============================================================
save_dir = '../app/ai_service/models'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/model_routing.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open(f'{save_dir}/label_encoder_agent.pkl', 'wb') as f:
    pickle.dump(le_agent, f)

with open(f'{save_dir}/routing_rules.pkl', 'wb') as f:
    pickle.dump(routing_rules, f)

print("✅ Modèles routage sauvegardés dans : backend/app/ai_service/models/")
for fname in sorted(os.listdir(save_dir)):
    if fname.endswith('.pkl'):
        size = os.path.getsize(f'{save_dir}/{fname}')
        print(f"   {fname:<40} {size/1024:.1f} KB")

---
## 10. Résumé

In [ ]:
# ============================================================
# RÉSUMÉ FINAL
# ============================================================
print("=" * 65)
print("  RÉSUMÉ — ROUTAGE AGENT HELPDESK IA GEISER")
print("=" * 65)

print(f"\n📊 APPROCHE 1 — ML + RÈGLES HYBRIDE")
print(f"   Modèle          : {best_name}")
print(f"   Accuracy        : {results[best_name]['accuracy']*100:.2f}%")
print(f"   F1-Score        : {results[best_name]['f1_score']:.4f}")
print(f"   Nb agents       : {len(le_agent.classes_)}")
print(f"   Seuil confiance : 60%")
print(f"   Stratégies      : ML → Règle catégorie → Round-robin")

print(f"\n📊 APPROCHE 2 — SCORING MULTI-CRITÈRES")
print(f"   Formule         : (expertise × 2) - charge_actuelle")
print(f"   Critères        : expertise historique + charge de travail")
print(f"   Avantage        : transparent, adaptatif, sans entraînement")

print(f"\n📋 COMPARAISON")
print(f"   ML + Règles  → meilleur si on a beaucoup d'historique")
print(f"   Scoring      → meilleur si on veut équilibrer la charge")

print(f"\n✅ Fichiers sauvegardés :")
print(f"   model_routing.pkl, label_encoder_agent.pkl, routing_rules.pkl")

print(f"\n🚀 Prochaine étape :")
print(f"   Notebook 04 — RAG Pipeline (ChromaDB + Suggestions)")
print("=" * 65)